# Notebook 04: Integración Multimodal y Síntesis Pericial con LLM

## Contexto y Objetivo
En este notebook se prototipa la conexión entre las salidas cuantitativas de los modelos de visión artificial y un Modelo de Lenguaje Grande (LLM - GPT-3.5-Turbo).

### Pipeline de Síntesis:
1. **Entrada Estructurada:** Fusión de metadata administrativa del siniestro (número, fecha, ubicación, declaración) con las probabilidades de daño y detecciones de piezas.
2. **Prompt Engineering Contextual:** Instrucciones estrictas para evitar alucinaciones, garantizando que el reporte técnico solo describa daños fundamentados en las predicciones visuales.
3. **Salida Pericial:** Minuta en lenguaje natural apta para la toma de decisiones del perito o liquidador de seguros.


In [4]:
import openai
import config
from openai import OpenAI

In [7]:
client = OpenAI(api_key=config.OPENAI_API_KEY)

In [5]:
data = {
    "siniestro": {
        "numero": "SIN123456789",
        "fecha": "2024-11-03T15:30:00Z",
        "ubicacion": {
            "direccion": "Av. Principal 123",
            "pais": "País Ejemplo"
        },
        "descripcion": "Colisión con otro vehículo en un cruce de semáforo."
    },
    "damaged_scores": {
        "damaged": 0.33178362250328064,
        "good": 0.3553612530231476,
        "unknown": 0.3128550946712494
    },
    "detected_objects": [
        {
            "label": "damage",
            "score": 0.9820688366889954,
            "box": [
                366.3293762207031, 242.73886108398438, 774.0194091796875,
                599.7211303710938
            ]
        },
        {
            "label": "damage",
            "score": 0.6540231704711914,
            "box": [
                715.1776123046875, 642.21875, 971.5934448242188, 795.8441772460938
            ]
        }
    ]
}

In [34]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages= [{
        "role": "user",
        "content": f"Te voy a dar un JSON, quiero que lo analices y brindes una respuesta adecual general que sea natural. Siniestro es la información del hecho, damaged_scores y detected_objects son respuestas de un clasificador y un detector. La respuesta debe cumplir con ciertas condiciones: Si la clase máxima en damaged scores es damaged, analizamos que hayan cosas en damaged objets y coincida con la observación, caos contrario, si es good no hubo siniestro y si es unknown, la imagen no corresponde a nada conocido. No hace falta que seas explicito en la respuesta con las observaciones. El JSON es:\n{data}"
    }]
)

In [54]:
completion.choices[0].message.content

'Se ha detectado un siniestro con el número SIN123456789, ocurrido el 3 de noviembre de 2024 a las 15:30 en la dirección Av. Principal 123, País Ejemplo. Según los resultados del clasificador, se observa que hay daños en los objetos detectados, lo que coincide con la descripción del siniestro.'